# 04 — Agent + RAG + Streamlit Demo

Wire everything together:
1. Build a FAISS retrieval index over all 70k+ ICD-10 codes
2. Implement the 4 tools from `src/tools.py`
3. Build a custom tool-calling agent loop (no LangChain)
4. Test the agent on the 5 demo doctor's notes
5. Run a 5-configuration ablation eval
6. Launch the Streamlit app via ngrok

## Prerequisites
- Notebook 3 completed (`models/merged-q4.gguf` exists)
- Codebook from Notebook 1 (`data/processed/icd10_codebook.parquet`)

## Runtime
~20-40 min total. CPU is fine.

## The core insight

This notebook is where we prove the thesis: **no single technique wins alone**.
You'll see:

- **Base model**: bad format, low accuracy
- **Fine-tuned (Q4)**: good format, but hallucinates rare codes
- **+ RAG**: broader coverage
- **+ Tools**: no hallucination (invalid codes caught)


## 0. Bootstrap

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = (
    'COLAB_GPU' in os.environ
    or 'COLAB_RELEASE_TAG' in os.environ
    or (Path('/content').exists() and not Path('/content').is_symlink())
)

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/icd10-slm')
else:
    PROJECT = Path.cwd()
    while not (PROJECT / 'requirements.txt').exists():
        if PROJECT.parent == PROJECT:
            raise RuntimeError(f"Couldn't find repo root from {Path.cwd()}")
        PROJECT = PROJECT.parent

assert PROJECT.exists(), f"Project not found at {PROJECT}"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Load secrets — .env from /content (Path B) or from project
if not os.getenv('HF_TOKEN'):
    try:
        from dotenv import load_dotenv
    except ImportError:
        !pip install -q python-dotenv
        from dotenv import load_dotenv
    import shutil
    candidates = [
        Path('/content/.env'),
        Path('/content/drive/MyDrive/icd10-slm/.env'),
    ]
    for c in candidates:
        if c.exists():
            if c != Path('/content/.env'):
                shutil.copy(c, '/content/.env')
            load_dotenv('/content/.env')
            break

assert os.getenv('HF_TOKEN'), "HF_TOKEN missing"
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

from src import config
print(f"✓ env:     {'Colab' if IN_COLAB else 'Local'}")
print(f"✓ project: {PROJECT}")
print(f"✓ model:   {config.BASE_MODEL}")


In [ ]:
# Verify prerequisites
assert config.GGUF_PATH.exists(), "Q4 GGUF missing — run Notebook 3 first"
assert config.CODEBOOK_PATH.exists(), "Codebook missing — run Notebook 1 first"
print(f"✓ Q4 model:  {config.GGUF_PATH.stat().st_size/1024/1024:.1f} MB")

import pandas as pd
codebook = pd.read_parquet(config.CODEBOOK_PATH)
print(f"✓ Codebook: {len(codebook):,} codes (from Notebook 1)")


## 1. Expand codebook to full ICD-10 scope for RAG

The codebook from Notebook 1 has only the ~1,564 codes we trained on. For RAG
to be useful, the retriever needs to know about ALL codes in the original
dataset (so it can suggest codes the model has never seen).

We reload `krishnareddy/icddxdescmap` for this broader set.


In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd

splits = load_dataset("krishnareddy/icddxdescmap")
raw = concatenate_datasets([splits[s] for s in splits.keys()])
df = raw.to_pandas()

# Normalize columns (same defensive rename as Notebook 1)
rename_map = {}
for col in df.columns:
    low = col.lower()
    if low in ('annotationstring', 'docdesc'):    rename_map[col] = 'clinical_text'
    elif low in ('dxcode', 'code'):                rename_map[col] = 'code'
    elif low in ('shortdesc', 'short_desc'):       rename_map[col] = 'short_desc'
    elif low in ('longdesc', 'long_desc'):         rename_map[col] = 'long_desc'
df = df.rename(columns=rename_map)

# Build the full codebook
full_codebook = (
    df[['code', 'short_desc', 'long_desc']]
    .drop_duplicates('code').sort_values('code').reset_index(drop=True)
)
full_codebook['category'] = full_codebook['code'].str[:3]

print(f"✓ Full codebook: {len(full_codebook):,} codes")


## 2. Build FAISS index over full codebook

Each code is embedded as `<code>: <short_desc> — <long_desc>`. At query time
we embed the clinical description and find the top-k closest codes by cosine similarity.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(config.EMBEDDING_MODEL)

# Build combined text for each code
full_codebook['rag_text'] = (
    full_codebook['code'] + ': ' +
    full_codebook['short_desc'] + ' — ' +
    full_codebook['long_desc']
)

print(f"Embedding {len(full_codebook):,} codes...")
embeddings = embedder.encode(
    full_codebook['rag_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # for cosine via inner product
)
print(f"✓ embeddings shape: {embeddings.shape}")


In [ ]:
import faiss

d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)  # inner product == cosine (since normalized)
index.add(embeddings.astype(np.float32))

# Save to disk
config.RAG_INDEX_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(config.RAG_INDEX_DIR / 'icd10.faiss'))
full_codebook.to_parquet(config.RAG_INDEX_DIR / 'codebook.parquet', index=False)

print(f"✓ FAISS index saved to {config.RAG_INDEX_DIR}")

# Test a query
def search(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_emb, k)
    return [
        {
            'code': full_codebook.iloc[idx]['code'],
            'short_desc': full_codebook.iloc[idx]['short_desc'],
            'score': float(scores[0][i]),
        }
        for i, idx in enumerate(idxs[0])
    ]

print("\n--- test query: 'patient has Type 2 diabetes' ---")
for hit in search("patient has Type 2 diabetes", k=5):
    print(f"  {hit['score']:.3f}  {hit['code']:8s}  {hit['short_desc'][:55]}")


## 3. Implement the 4 tools

These operate against the full codebook. The agent will call these during its
reasoning loop.


In [ ]:
# --- Tool state ---
CODE_LOOKUP = full_codebook.set_index('code').to_dict(orient='index')
VALID_CODES = set(full_codebook['code'])

# ICD-10 chapter ranges (simplified)
CHAPTER_RANGES = [
    ('A00-B99', 'Infectious and parasitic'),
    ('C00-D49', 'Neoplasms'),
    ('D50-D89', 'Blood and immune disorders'),
    ('E00-E89', 'Endocrine, nutritional, metabolic'),
    ('F01-F99', 'Mental, behavioral, neurodevelopmental'),
    ('G00-G99', 'Nervous system'),
    ('H00-H59', 'Eye and adnexa'),
    ('H60-H95', 'Ear and mastoid'),
    ('I00-I99', 'Circulatory system'),
    ('J00-J99', 'Respiratory system'),
    ('K00-K95', 'Digestive system'),
    ('L00-L99', 'Skin and subcutaneous tissue'),
    ('M00-M99', 'Musculoskeletal'),
    ('N00-N99', 'Genitourinary'),
    ('O00-O9A', 'Pregnancy, childbirth'),
    ('P00-P96', 'Perinatal'),
    ('Q00-Q99', 'Congenital malformations'),
    ('R00-R99', 'Symptoms, signs, abnormal findings'),
    ('S00-T88', 'Injury, poisoning'),
    ('V00-Y99', 'External causes'),
    ('Z00-Z99', 'Factors influencing health'),
]

def icd10_lookup(code: str) -> dict:
    code = code.strip().upper()
    if code in CODE_LOOKUP:
        info = CODE_LOOKUP[code]
        return {
            'code': code,
            'short_desc': info['short_desc'],
            'long_desc': info['long_desc'],
            'is_valid': True,
        }
    return {'code': code, 'short_desc': '', 'long_desc': '', 'is_valid': False}

def icd10_search(query: str, k: int = 10) -> list[dict]:
    return search(query, k)

def validate_code(code: str) -> bool:
    return code.strip().upper() in VALID_CODES

def code_hierarchy(code: str) -> dict:
    code = code.strip().upper()
    cat = code[:3]
    letter = code[0] if code else ''
    chapter = 'Unknown'
    for rng, name in CHAPTER_RANGES:
        start, end = rng.split('-')
        if start[0] == letter:
            chapter = name
            break
    return {'code': code, 'category': cat, 'chapter': chapter}


# Smoke tests
print("Tool smoke tests:")
print(f"  validate_code('E11.9'):     {validate_code('E11.9')}")
print(f"  validate_code('FAKE'):      {validate_code('FAKE')}")
print(f"  lookup('E11.9'):            {icd10_lookup('E11.9')['short_desc']}")
print(f"  search('hypertension'):     {icd10_search('hypertension', 3)[0]['code']}")
print(f"  hierarchy('E11.9'):         {code_hierarchy('E11.9')['chapter']}")


## 4. Load Q4 model (same as Notebook 3)

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=str(config.GGUF_PATH),
    n_ctx=2048,
    n_threads=4,
    verbose=False,
)

def llm_raw(messages, max_tokens=16, temperature=0):
    r = llm.create_chat_completion(messages=messages, max_tokens=max_tokens, temperature=temperature)
    return r['choices'][0]['message']['content'].strip()

print("✓ Q4 model loaded")


## 5. Custom agent loop

The agent:
1. Gets candidate codes from RAG
2. Asks Q4 model to pick one
3. Validates the pick against the codebook
4. If invalid → calls `icd10_search` as fallback, retries
5. Returns the best valid code it found


In [ ]:
def agent_code(clinical_text: str, use_rag: bool = True, use_tools: bool = True) -> dict:
    steps = []

    # --- Step 1: RAG retrieval --------------------------------
    if use_rag:
        candidates = icd10_search(clinical_text, k=config.RAG_TOP_K)
        context = "Candidate codes (most relevant first):\n" + "\n".join(
            f"  {c['code']}: {c['short_desc']}" for c in candidates
        )
        steps.append({'tool': 'icd10_search', 'result': [c['code'] for c in candidates[:5]]})
    else:
        context = ""

    # --- Step 2: Ask model for a code --------------------------
    user_prompt = clinical_text
    if context:
        user_prompt = context + "\n\nClinical description: " + clinical_text

    raw = llm_raw([
        {"role": "system", "content": config.SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ])
    steps.append({'step': 'model_predict', 'output': raw})

    # --- Step 3: Validate with tool ----------------------------
    if use_tools:
        if validate_code(raw):
            final = raw.strip().upper()
        else:
            # Fallback: pick the top RAG hit
            steps.append({'step': 'validation_failed', 'fell_back_to_rag': True})
            if use_rag and candidates:
                final = candidates[0]['code']
            else:
                # No RAG available — do an emergency search
                fallback = icd10_search(clinical_text, k=1)
                final = fallback[0]['code'] if fallback else raw.strip().upper()
    else:
        final = raw.strip().upper()

    return {
        'code': final,
        'description': CODE_LOOKUP.get(final, {}).get('short_desc', '(unknown)'),
        'steps': steps,
    }


# Test on one example
print("--- agent test: 'patient has Type 2 diabetes' ---")
res = agent_code("patient has Type 2 diabetes without complications")
print(f"Final code: {res['code']}  ({res['description']})")
for s in res['steps']:
    print(f"  step: {s}")


## 6. Run the agent on the 5 demo doctor's notes

For each note, chunk it into phrases, run the agent on each phrase, dedupe
the codes. This is exactly what the Streamlit app will do.


In [ ]:
from src.note_chunker import chunk_note
from app.demo_notes import DEMO_NOTES

def code_note(note_text: str, use_rag=True, use_tools=True) -> list[dict]:
    phrases = chunk_note(note_text)
    codes_seen = {}
    for p in phrases:
        r = agent_code(p, use_rag=use_rag, use_tools=use_tools)
        c = r['code']
        if c and validate_code(c):
            codes_seen[c] = r['description']
    return [{'code': c, 'description': d} for c, d in codes_seen.items()]


# Test on case 1
key = 'case_01_t2dm_htn'
note_data = DEMO_NOTES[key]
print(f"=== {note_data['title']} ===")
print(f"Expected codes: {note_data['expected_codes']}")

predicted = code_note(note_data['note'])
print(f"\nPredicted codes:")
for p in predicted:
    hit = '✓' if p['code'] in note_data['expected_codes'] else ' '
    print(f"  {hit} {p['code']:8s} {p['description']}")


## 7. Ablation eval — 5 configurations

Run each configuration on the same 200-example test sample. Shows which
components contribute to final performance.

Configurations:
1. **Base-Q4** (no RAG, no tools) — quality baseline
2. **Base-Q4 + RAG** (retrieval only)
3. **Base-Q4 + Tools** (validation only)
4. **Base-Q4 + RAG + Tools** (full agent) ← our headline result


In [ ]:
import json, random
from src.eval import evaluate

with open(config.TEST_PATH) as f:
    test_rows = [json.loads(line) for line in f]

random.seed(42)
test_sample = random.sample(test_rows, 200)
test_inputs = [{'input': ex['clinical_text'], 'gold': ex['code']} for ex in test_sample]

configs = [
    ('Q4 only',          False, False),
    ('Q4 + RAG',         True,  False),
    ('Q4 + Tools',       False, True),
    ('Q4 + RAG + Tools', True,  True),
]

results = []
for name, use_rag, use_tools in configs:
    print(f"Evaluating: {name}")
    def pred_fn(text):
        return agent_code(text, use_rag=use_rag, use_tools=use_tools)['code']
    res = evaluate(pred_fn, test_inputs, VALID_CODES, name)
    results.append(res)
    print(f"  exact: {res.exact_match*100:5.1f}%  category: {res.category_match*100:5.1f}%  "
          f"valid_code: {res.valid_in_codebook*100:5.1f}%")


In [ ]:
# Build the ablation table
ablation = pd.DataFrame([
    {
        'Configuration': r.config,
        'Exact match':     f"{r.exact_match*100:.1f}%",
        'Category match':  f"{r.category_match*100:.1f}%",
        'Valid code':      f"{r.valid_in_codebook*100:.1f}%",
        'Latency (ms)':    f"{r.avg_latency_ms:.0f}",
    }
    for r in results
])
print("\n🎯 ABLATION RESULTS (200 test examples)")
print("=" * 70)
print(ablation.to_string(index=False))

# Save
(config.MODELS_DIR / 'ablation_results.json').write_text(
    ablation.to_json(orient='records', indent=2)
)


## 8. Launch Streamlit app via ngrok

Starts the app on a public URL your students can access.

⚠️ **Keep this notebook cell running** for the app to stay alive. Stop it
(interrupt) when you're done demoing.


In [ ]:
# Install streamlit + pyngrok if missing
!pip install -q streamlit pyngrok

# The streamlit app is at app/streamlit_app.py — but it's a skeleton from Notebook 1.
# We need to update it to actually use our agent + model. Rewrite it here:

streamlit_script = '''
import sys, os, json
from pathlib import Path

# Find project root
PROJECT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT))

import streamlit as st
import pandas as pd
from src import config
from src.note_chunker import chunk_note
from app.demo_notes import DEMO_NOTES

st.set_page_config(page_title="ICD-10 Coding SLM", layout="wide")
st.error("⚠️  Educational demo only — not medical advice. No real patient data.")
st.title("ICD-10 Coding SLM — Fine-tuned Q4 + Agent")

# Cache the heavy model load
@st.cache_resource
def load_agent():
    from llama_cpp import Llama
    import faiss
    from sentence_transformers import SentenceTransformer
    import numpy as np

    codebook = pd.read_parquet(config.RAG_INDEX_DIR / 'codebook.parquet')
    index = faiss.read_index(str(config.RAG_INDEX_DIR / 'icd10.faiss'))
    embedder = SentenceTransformer(config.EMBEDDING_MODEL)
    llm = Llama(model_path=str(config.GGUF_PATH), n_ctx=2048, n_threads=4, verbose=False)
    valid = set(codebook['code'])
    lookup = codebook.set_index('code').to_dict(orient='index')
    return llm, embedder, index, codebook, valid, lookup

with st.spinner("Loading model + RAG index..."):
    llm, embedder, index, codebook, VALID, LOOKUP = load_agent()

# Case selector
case_key = st.selectbox(
    "Choose a sample clinical note:",
    options=list(DEMO_NOTES.keys()),
    format_func=lambda k: DEMO_NOTES[k]["title"],
)
note_data = DEMO_NOTES[case_key]
st.text_area("Clinical note:", value=note_data["note"], height=250, disabled=True)

if st.button("🔬 Code this note", type="primary"):
    import numpy as np

    phrases = chunk_note(note_data["note"])
    st.markdown(f"**Chunked into {len(phrases)} phrases**")

    codes_found = {}
    progress = st.progress(0, text="Coding phrases...")

    for i, phrase in enumerate(phrases):
        # RAG
        q_emb = embedder.encode([phrase], normalize_embeddings=True).astype(np.float32)
        scores, idxs = index.search(q_emb, 10)
        candidates = [codebook.iloc[idx] for idx in idxs[0]]
        ctx = "Candidates:\n" + "\n".join(f"  {c['code']}: {c['short_desc']}" for c in candidates[:5])

        # Model call
        r = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": config.SYSTEM_PROMPT},
                {"role": "user",   "content": ctx + "\n\nClinical: " + phrase},
            ],
            max_tokens=16, temperature=0,
        )
        pred = r['choices'][0]['message']['content'].strip().upper()

        # Validate
        if pred not in VALID:
            pred = candidates[0]['code']  # fall back to top RAG hit

        if pred not in codes_found:
            codes_found[pred] = {
                'phrase': phrase,
                'desc': LOOKUP.get(pred, {}).get('short_desc', '(unknown)'),
            }

        progress.progress((i + 1) / len(phrases), text=f"Coding phrase {i+1}/{len(phrases)}")

    progress.empty()

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("✅ Predicted Codes")
        for code, info in codes_found.items():
            expected = code in note_data["expected_codes"]
            badge = "🎯" if expected else "  "
            st.markdown(f"{badge} **{code}** — {info['desc']}")
            with st.expander("From phrase"):
                st.write(info['phrase'])

    with col2:
        st.subheader("📋 Expected (author annotation)")
        for code in note_data["expected_codes"]:
            found = code in codes_found
            st.markdown(f"{'✓' if found else '○'} {code}")

        predicted_set = set(codes_found.keys())
        expected_set = set(note_data["expected_codes"])
        overlap = predicted_set & expected_set
        st.markdown(f"**Match:** {len(overlap)} / {len(expected_set)}")
'''

(PROJECT / 'app' / 'streamlit_app.py').write_text(streamlit_script)
print("✓ streamlit app written to app/streamlit_app.py")


In [ ]:
# Set up ngrok and launch
from pyngrok import ngrok, conf
import subprocess, time

# You'll need a free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken
# Add to .env as NGROK_TOKEN, then:
ngrok_token = os.environ.get('NGROK_TOKEN')
if ngrok_token:
    conf.get_default().auth_token = ngrok_token

# Kill any existing tunnels
ngrok.kill()

# Start streamlit in the background
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', str(PROJECT / 'app' / 'streamlit_app.py'),
     '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(5)

# Open a tunnel
public_url = ngrok.connect(8501, "http")
print(f"\n🌐 Streamlit app live at: {public_url}")
print("\nShare this URL with anyone to let them try the demo.")
print("Stop the app: interrupt this cell (keyboard ⏹ above the cell output).")


## ✅ Done — project complete

### What you've built

A full pipeline from raw HuggingFace dataset → fine-tuned + quantized model →
retrieval-augmented agent → deployable Streamlit demo. Everything reproducible.

### Teaching takeaways

Refer to `docs/teaching_notes.md` for discussion prompts. Key points:

1. **Fine-tuning teaches format** — base model was bad at producing valid
   ICD-10 code strings; the fine-tuned model always does.
2. **RAG provides coverage** — the model knows ~1,564 codes but RAG helps it
   suggest codes it never saw.
3. **Tools prevent hallucination** — validation against the real codebook
   catches the few invalid codes the model still produces.
4. **Quantization enables deployment** — Q4 is 3× smaller, fits on Pi-class
   hardware, runs at interactive speeds on CPU.

### Commit + push everything

```bash
git add notebooks/ app/streamlit_app.py
git commit -m "Complete project: NB2 training + NB3 quantization + NB4 agent"
git push
```

### The ablation table and benchmark results are your presentation centerpiece.
